# Lab 1 — Environment Boot & First Tables

Welcome to the Apache Iceberg bootcamp! In this first lab you will:

1. Start a **Spark session** that is pre-wired to an Iceberg catalog.
2. Create a **database** and your **first Iceberg table**.
3. Insert and query data with plain SQL.
4. Look *under the hood* in MinIO to see what Iceberg actually writes to object storage.

**The stack you booted with `docker compose up`:**

| Service | Role | Where |
|---|---|---|
| MinIO | S3-compatible object storage — holds all table data & metadata | `http://localhost:9001` (console) |
| PostgreSQL | Backing database for the Hive Metastore | `localhost:5432` |
| Hive Metastore | The **catalog**: maps table names → current metadata file | `thrift://hive-metastore:9083` |
| Spark + Jupyter | Compute engine — where you are right now | `http://localhost:8888` |

> **Key mental model:** Iceberg is *not* a service you connect to. It is a **table format** — a spec for metadata files living next to your data in object storage. The catalog (Hive Metastore) essentially stores the table's registration plus a pointer to its current metadata file (a `metadata_location` property); the full table state — schema history, snapshots, file lists — lives in Iceberg's own metadata files in the bucket. Spark does the rest.

## Step 1 — Initialize the Spark session

All the Iceberg configuration (catalog definition, JARs, MinIO credentials) lives in `spark-defaults.conf`, which is mounted into this container at `$SPARK_HOME/conf/`. That means a plain `SparkSession.builder.getOrCreate()` is enough — **no config in code**.

The catalog is named **`hive_prod`** and is set as the default catalog, so `db.table` resolves to `hive_prod.db.table`.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Lab01-FirstTables").getOrCreate()

print(f"Spark version   : {spark.version}")
print(f"Default catalog : {spark.conf.get('spark.sql.defaultCatalog')}")
print(f"Catalog impl    : {spark.conf.get('spark.sql.catalog.hive_prod')}")
print(f"Metastore URI   : {spark.conf.get('spark.sql.catalog.hive_prod.uri')}")

ModuleNotFoundError: No module named 'pyspark'

### Sanity check: can Spark reach the catalog?

`SHOW DATABASES` is served by the Hive Metastore over its Thrift API — if this cell returns (even just the `default` database), your Spark ↔ Metastore ↔ PostgreSQL chain is working.

In [ ]:
spark.sql("SHOW DATABASES").show()

## Step 2 — Create a database

A *database* (a.k.a. *namespace*) is just a logical grouping of tables registered in the catalog. The metastore will also create a directory for it in the warehouse bucket: `s3a://warehouse/db.db/` (Hive convention: database directories get a `.db` suffix).

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS hive_prod.db")

spark.sql("SHOW DATABASES").show()

## Step 3 — Create your first Iceberg table

The magic words are **`USING iceberg`** — that tells Spark to create this table in the Iceberg format (managed by our Iceberg catalog) instead of as a plain Hive/Parquet table.

Even before any data is inserted, Iceberg writes a **metadata file** (`00000-<uuid>.metadata.json`) to object storage containing the schema, partition spec, and an empty snapshot history.

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS hive_prod.db.test_table (
        id   INT,
        name STRING
    )
    USING iceberg
""")

spark.sql("DESCRIBE TABLE EXTENDED hive_prod.db.test_table").show(50, truncate=False)

## Step 4 — Insert data

Every write to an Iceberg table produces a new **snapshot**: new Parquet data files + a new manifest + a new metadata file. Nothing is ever modified in place — this is the foundation for time travel (Lab 3).

In [ ]:
spark.sql("""
    INSERT INTO hive_prod.db.test_table
    VALUES (1, 'Alice'), (2, 'Bob')
""")

## Step 5 — Query the table

Reading works like any other Spark table — Iceberg resolves the current snapshot via the catalog, prunes files using metadata, and hands Spark the Parquet files to scan.

In [ ]:
df = spark.table("hive_prod.db.test_table")
df.show()

# The same thing in SQL:
spark.sql("SELECT * FROM hive_prod.db.test_table ORDER BY id").show()

## 🔍 Step 6 — Verify in MinIO: look under the hood

**Do this now — it is the most important part of the lab.**

1. Open the MinIO console at **http://localhost:9001** and log in with `admin` / `password`.
2. Go to **Object Browser → warehouse** and navigate into **`db.db/test_table/`**.
3. You will find two folders:
   - **`data/`** — the actual rows, stored as Parquet files. You should see one (or two) small Parquet files from the insert above.
   - **`metadata/`** — the Iceberg brain 🧠:
     - `0000N-<uuid>.metadata.json` — table metadata (schema, snapshots, partition spec). On a fresh run there are **two versions** — `00000-*` from `CREATE TABLE`, `00001-*` from the `INSERT` — and every further commit adds one more.
     - `snap-*.avro` — the manifest **list** for each snapshot.
     - `*-m0.avro` — the **manifest** files tracking individual data files with their column statistics.

**Questions to discuss with your neighbor:**
- Why did the `INSERT` create a *new* `metadata.json` instead of editing the existing one?
- What does the Hive Metastore actually store, if all of this lives in the bucket?

## Bonus — Iceberg metadata tables

You don't have to click through MinIO to inspect metadata: every Iceberg table exposes it as queryable *metadata tables* (`.snapshots`, `.files`, `.history`, `.manifests`, ...). We'll use these heavily in Labs 3 and 4.

In [ ]:
# The snapshot created by our INSERT:
spark.sql("""
    SELECT snapshot_id, committed_at, operation, summary['added-records'] AS added_records
    FROM hive_prod.db.test_table.snapshots
""").show(truncate=False)

# The physical Parquet files behind the table:
spark.sql("""
    SELECT file_path, record_count, file_size_in_bytes
    FROM hive_prod.db.test_table.files
""").show(truncate=False)

## ✅ Wrap-up

You have:
- ✔ Booted a complete lakehouse stack (object storage + catalog + compute)
- ✔ Created an Iceberg table and written/read data with SQL
- ✔ Seen the physical layout: `data/` (Parquet) + `metadata/` (JSON + Avro) in object storage

**Next (Lab 2):** we'll ingest data from PostgreSQL (`source_db`) into Iceberg tables and evolve their schemas safely.

*Optional cleanup if you want to re-run this lab from scratch:*
```python
spark.sql("DROP TABLE IF EXISTS hive_prod.db.test_table PURGE")
```